# Antarctic lateral drag — manuscript analysis

I'll use this main notebook for the experiment record, analytical form functions, FIA comparisons, seasonal summaries and manuscript exports. `LD-performance.ipynb` owns the supporting FIP, mechanics, face-stress, occurrence, anchor-distance, pack and point-thickness analyses. I'll review its exported evidence here rather than duplicate those calculations.

Both notebooks use `ld_pub_config.json`, the same retained experiment definitions and the grouped AFIM/GRAPHICAL roots. Model diagnostics cover 2000–2005 after the 1995–1999 spin-up. Observational comparisons currently use 1 April 2000–30 April 2005 inclusive; I'll confirm that endpoint before final exports. All figures in this notebook use PyGMT.

Default settings run the inventory and write processing recipes/review records only. Scientific stages remain pending until their inputs and definitions are reviewed; none of the three new runs is treated as analysed merely because its directory exists.

## 0. Shared configuration and loaders

I'll use the Gadi checkouts under `~/src/shuga` and `~/src/CICE_free-slip/notebooks`, with imports configured before loading local modules. `SHUGA_REPO`, `CICE_FREE_SLIP_REPO`, `LD_PUB_CONFIG` and `LD_PUB_EXPORT` allow explicit overrides. The helper and configuration are the same files used by the performance notebook; running this notebook does not execute or modify that notebook.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shlex
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

# Match the Gadi checkout layout; environment overrides support other machines.
D_SRC = Path.home() / "src"
D_NB_ROOT = Path(os.environ.get("CICE_FREE_SLIP_REPO", D_SRC / "CICE_free-slip")) / "notebooks"
if not (D_NB_ROOT / "ld_pub_workflow.py").is_file():
    D_NB_ROOT = next((p for p in (Path.cwd(), Path.cwd()/"notebooks")
                      if (p/"ld_pub_workflow.py").is_file()), D_NB_ROOT)
D_shuga = Path(os.environ.get("SHUGA_REPO", D_SRC / "shuga"))
NOTEBOOK_DIR, SHUGA_REPO = D_NB_ROOT, str(D_shuga)
if not (NOTEBOOK_DIR/"ld_pub_workflow.py").is_file():
    raise FileNotFoundError(f"Publication helper missing from {NOTEBOOK_DIR}")
sys.path.insert(0, str(NOTEBOOK_DIR))
if (D_shuga/"__init__.py").is_file():
    sys.path.insert(0, str(D_shuga.parent))
import ld_pub_workflow as wf
import shuga
from shuga.core.types import RunSpec, ClassificationSpec, MetricsSpec
from shuga.core.paths import ShugaPaths
from shuga.core.regions import ANTARCTIC_8_REGIONS

In [2]:
CONFIG_PATH = Path(os.environ.get('LD_PUB_CONFIG', NOTEBOOK_DIR/'ld_pub_config.json'))
CFG = json.loads(CONFIG_PATH.read_text())
START, END = CFG['model_start'], CFG['model_end']
OBS_START, OBS_END = CFG['evaluation_start'], CFG['evaluation_end']
GROUP, METHOD = CFG['experiment_group'], CFG['method']
BANK = Path(CFG['afim_output_root'])
ARCHIVE = BANK
EXPORT = Path(os.environ.get('LD_PUB_EXPORT', str(Path(CFG['graphics_root'])/GROUP/'publication')))
EXPORT.mkdir(parents=True, exist_ok=True)
CACHE = EXPORT/'cache'; CACHE.mkdir(exist_ok=True)
manifest = wf.expected_experiments()
EXPERIMENTS = manifest.experiment.tolist()
FAMILIES = {family: part.experiment.tolist() for family, part in manifest.groupby('family', sort=False)}
REFERENCE = 'no-slip-LFI-match'
PRIMARY = ['no-slip-LFI-match','free-slip-no-drag','Cs-high','Cq-high']
TENSILE_CASES = ['Cs-high-ktens-zero','Cs-high-ktens-low','Cs-high-ktens-mid','Cs-high']
DIAGNOSTIC_CASES = list(dict.fromkeys(PRIMARY+TENSILE_CASES))
PAIRS = {'boundary_only':('free-slip-no-drag','no-slip-LFI-match'),
         'static_drag':('Cs-high','free-slip-no-drag'),
         'quadratic_drag':('Cq-high','free-slip-no-drag'),
         'tensile_term':('Cs-high','Cs-high-ktens-zero'),
         'coast_increment_over_gib':('Cs-high','GIB-drag'),
         'gib_increment_over_coast':('Cs-high','cst-drag')}
REGIONS = list(ANTARCTIC_8_REGIONS)
OBS_FIA_PATH = Path(CFG['obs_fia_path']).expanduser()
UNIT_OVERRIDES = {}  # Only documented metadata corrections; never inferred from values.
OPEN, COVERAGE, ARTIFACTS = {}, [], []

In [3]:
def contexts(name, method=METHOD, ice_type='FI'):
    run=RunSpec(name,START,END,hemisphere='SH',project=CFG['project'],user=CFG['user'],experiment_group=GROUP)
    cls=ClassificationSpec(ice_type=ice_type,methods=method,grid_type=CFG['grid_type'],
        ispd_thresh=CFG['ispd_thresh'],aice_thresh=CFG['aice_thresh'],
        bin_window=CFG['bin_window'],bin_min_days=CFG['bin_min_days'],roll_window=CFG['roll_window'])
    met=MetricsSpec(methods=method)
    paths=ShugaPaths(run_cfg=run,cls_cfg=cls,met_cfg=met,afim_output_root=BANK,
                     archive_root=ARCHIVE,graphics_root=CFG['graphics_root'])
    return dict(run_cfg=run,cls_cfg=cls,met_cfg=met,pth_cfg=paths)

def load(name,kind,variables,method=METHOD,ice_type='FI'):
    key=(name,kind,tuple(variables),method,ice_type)
    if key not in OPEN:
        kwargs=contexts(name,method,ice_type)
        if kind!='history': kwargs['classification']=method
        loader={'metrics':shuga.load_metrics,'mask':shuga.load_classified,'history':shuga.load_cice}[kind]
        ds=loader(**kwargs,variables=variables,dt0_str=START,dtN_str=END)
        missing=set(variables)-set(ds.variables)
        if missing: ds.close(); raise KeyError(f'{name}: missing {missing}')
        OPEN[key]=ds
    return OPEN[key]

def open_product(path):
    path=Path(path)
    return xr.open_zarr(path,consolidated=False) if path.suffix=='.zarr' or path.is_dir() else xr.open_dataset(path)

def audited(da,name):
    da=da.copy(deep=False)
    if (name,da.name) in UNIT_OVERRIDES:
        da.attrs=dict(da.attrs,units=UNIT_OVERRIDES[(name,da.name)])
    return da

def git_commit(path):
    try:return subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
    except (OSError,subprocess.CalledProcessError):return 'UNVERIFIED'

def register(path,diagnostic,cases,period=None,notes=''):
    period=period or (OBS_START,OBS_END)
    ARTIFACTS.append(dict(path=str(path),diagnostic=diagnostic,experiments=list(cases),
        start=period[0],end=period[1],method=METHOD,signature=SIGNATURE,notes=notes))

def export_table(table,stem,cases,period=None):
    period=period or (OBS_START,OBS_END)
    path=EXPORT/f'{stem}_{period[0]}_{period[1]}.csv'
    table.to_csv(path,index=False);register(path,stem,cases,period)
    return path

In [4]:
provenance = dict(config                  = CFG,
                  helper_sha256           = wf.file_sha256(NOTEBOOK_DIR / 'ld_pub_workflow.py'),
                  model_repository_commit = git_commit(NOTEBOOK_DIR.parent),
                  shuga_module            = shuga.__file__,
                  shuga_commit            = git_commit(Path(shuga.__file__).parent),
                  unit_overrides          = {str(k):v for k,v in UNIT_OVERRIDES.items()})
SIGNATURE  = wf.signature(provenance)
print(f'Group: {GROUP}; model diagnostics: {START}–{END}; observational skill: {OBS_START}–{OBS_END}')
print('Inputs:',BANK/GROUP,'Outputs:',EXPORT)

Group: lateral-drag; model diagnostics: 2000-01-01–2005-12-31; observational skill: 2000-04-01–2005-04-30
Inputs: /g/data/gv90/da1339/afim_output/lateral-drag Outputs: /g/data/gv90/da1339/GRAPHICAL/lateral-drag/publication


### Main notebook settings

I'll set the following parameters before running the analysis stages. First run the shared configuration above. Leave `RUN_*` switches false for inventory-only work, then select the required cases and supply reviewed definitions. Review flags record decisions I've completed; setting them true does not perform a validation.

- `RUN_FIA` calculates matched observational comparisons; `RUN_FIA_PLOTS` replots the reduced FIA series. `RUN_ROBUSTNESS` additionally requires metrics for every method in `ROBUSTNESS_METHODS`.
- `RUN_ANALYTICAL` requires reviewed form-function equations, axis units, archived coefficients and `REGULARISATION_SPEED` values.
- `FIA_OBS_MODE` chooses explicit daily interpolation or exact native observation dates. I'll review the composite support in either mode, and approve interpolation separately when used.
- `GROWTH_WINDOW`, `RETREAT_WINDOW` and `MIN_EDGE_TRIM_DAYS` define the seasonal/minimum summaries. Classification padding and observation sampling require their own checks.
- `ACTIVE_EXPERIMENTS` may be a labelled exploratory subset. Final readiness requires the full retained set, including all three new comparisons.
- I'll update `MANUSCRIPT_CHECKS` after reviewing the companion exports, figures and text. `FINAL_EXPORT=True` enforces that review rather than inferring readiness from successful execution.

I'll restart and run from the top after changing configuration, avoiding mixed interactive state. Expensive conversion/classification and metric generation belong in allocated jobs; recipe creation here does not submit or execute them.


In [5]:
PENDING_EXPERIMENTS = {"free-slip-no-drag", "Cs-high-ktens-zero", "no-slip-LFI-match"}
ACTIVE_EXPERIMENTS  = [name for name in EXPERIMENTS if name not in PENDING_EXPERIMENTS]

### Main notebook settings

I'll set the following parameters before running the analysis stages. First run the shared configuration above. Leave `RUN_*` switches false for inventory-only work, then select the required cases and supply reviewed definitions. Review flags record decisions I've completed; setting them true does not perform a validation.

- `RUN_FIA` calculates matched observational comparisons; `RUN_FIA_PLOTS` replots the reduced FIA series. `RUN_ROBUSTNESS` additionally requires metrics for every method in `ROBUSTNESS_METHODS`.
- `RUN_ANALYTICAL` requires reviewed form-function equations, axis units, archived coefficients and `REGULARISATION_SPEED` values.
- `FIA_OBS_MODE` chooses explicit daily interpolation or exact native observation dates. I'll review the composite support in either mode, and approve interpolation separately when used.
- `GROWTH_WINDOW`, `RETREAT_WINDOW` and `MIN_EDGE_TRIM_DAYS` define the seasonal/minimum summaries. Classification padding and observation sampling require their own checks.
- `ACTIVE_EXPERIMENTS` may be a labelled exploratory subset. Final readiness requires the full retained set, including all three new comparisons.
- I'll update `MANUSCRIPT_CHECKS` after reviewing the companion exports, figures and text. `FINAL_EXPORT=True` enforces that review rather than inferring readiness from successful execution.

I'll restart and run from the top after changing configuration, avoiding mixed interactive state. Expensive conversion/classification and metric generation belong in allocated jobs; recipe creation here does not submit or execute them.


In [ ]:
# Analysis stages; default inventory runs do not open scientific products.
RUN_FIA = False                         # Load FIA products and calculate matched observational comparisons.
RUN_ANALYTICAL = False                  # Plot the reviewed static/quadratic analytical form functions in PyGMT.
RUN_ROBUSTNESS = False                  # Compare FIA skill under raw, binary-days and rolling-mean classifications.
RUN_FIA_PLOTS = False                   # Plot monthly FIA climatology with PyGMT after RUN_FIA.
FIA_OBS_MODE = "linear-daily"            # Preserve prior policy; alternative: 'native-dates' (no interpolation).
OBS_FIA_TEMPORAL_SUPPORT_VERIFIED = False # Review how composite observations relate to model daily samples.
OBS_FIA_INTERPOLATION_VERIFIED = False   # Required only for the explicit linear-daily observation mode.
FIA_DEFINITION_VERIFIED = False         # Check concentration-weighted FIA, units and observation compatibility.
FORM_FUNCTION_DEFINITION_VERIFIED = False # Check the equations against the executed CICE version.
FORM_FUNCTION_UNITS = None              # Reviewed common units for phi; required for analytical axis labelling.
GEOMETRY_MATCH_VERIFIED = False         # Review forcing, physics and form-factor provenance for geometry contrasts.
PROVENANCE_REVIEWED = False             # Review executable, forcing and restart lineage independently of file presence.
FINAL_EXPORT = False                    # Require all scientific and manuscript review items before declaring readiness.
REGIONAL_PARTITION_VERIFIED = False     # Declare only after checking sector boundaries for overlap and gaps.
REGIONAL_SUM_TOLERANCE = 1e-5            # Absolute tolerance in 10^3 km^2; relevant only to a verified partition.
REGULARISATION_SPEED = {}               # Experiment -> verified static u0 [m/s] from the executed implementation.
ACTIVE_EXPERIMENTS = list(EXPERIMENTS)  # Use all retained cases for final exports; subsets remain exploratory.
FIA_PLOT_CASES = list(PRIMARY)           # Experiments shown alongside AF2020 in the FIA climatology.
ROBUSTNESS_METHODS = ["raw", "binary-days", "rolling-mean"] # Precomputed classifications to compare.
GROWTH_WINDOW = (4, 7)                  # Inclusive April-July season; overrides the FIA CLI's Apr-Jun default.
RETREAT_WINDOW = (12, 3)                # Inclusive December-March season, crossing into the next year.
MIN_EDGE_TRIM_DAYS = 0                  # Summary-minimum trimming; review classification padding before changing.
MANUSCRIPT_CHECKS = {                   # Manual review, not inferred from successful computation.
    "companion_diagnostics_reviewed": False,
    "legacy_case_identity_resolved": False,
    "figures_captions_and_regional_signs_checked": False,
    "discussion_conclusions_abstract_complete": False,
    "authorship_funding_code_data_statements_checked": False,
    "full_manuscript_compiled_and_visually_checked": False,
}
if not ACTIVE_EXPERIMENTS or set(ACTIVE_EXPERIMENTS)-set(EXPERIMENTS):
    raise ValueError("Choose a non-empty subset of the retained experiment manifest")

def season_months(window):
    start,end=window
    if not (1<=start<=12 and 1<=end<=12):raise ValueError('Season months must be 1..12')
    return list(range(start,end+1)) if end>=start else list(range(start,13))+list(range(1,end+1))

provenance['publication_settings']=dict(
    observation_support=FIA_OBS_MODE,active_experiments=ACTIVE_EXPERIMENTS,
    growth_months=season_months(GROWTH_WINDOW),retreat_months=season_months(RETREAT_WINDOW),
    minimum_trim_days=MIN_EDGE_TRIM_DAYS,robustness_methods=ROBUSTNESS_METHODS,
    regularisation_speed=REGULARISATION_SPEED,form_function_units=FORM_FUNCTION_UNITS)
if MIN_EDGE_TRIM_DAYS<0:raise ValueError('Minimum edge trim must be non-negative')
SIGNATURE=wf.signature(provenance)


## 1. Experiment and provenance record

The expected settings below come from the reviewed namelists, not directory names. `no-slip-LFI` has ellipse ratios 1.2; `no-slip-LFI-match` has 1.5. `Cs-high-def-rhe` also changes EVP subcycling to 720 and is not an isolated tensile test. The two drag-off controls retain nonzero stored Cs but disable the drag switch. No new high-tensile case is implied by the unresolved legacy label `Cs-high-ktens-high`.

I'll check the executed namelists, model/analysis commits, grid and geometry versions/checksums, forcing versions, restart lineage, spin-up, actual daily coverage and history output definitions. Multiple archived namelists require an explicit selection; an archive filename alone does not prove that it governed all history segments. The old archive dry run found no history in the three case directories, so actual job output locations still need confirming.

In [ ]:
NAMELIST_SELECTION={}  # Explicit experiment -> archived ice_in path if multiple records exist.
rows=[]
for row in manifest.to_dict('records'):
    name=row['experiment']; paths=contexts(name)['pth_cfg']; root=paths.output_root
    candidates=sorted((root/'run_metadata').glob('*/ice_in'))
    path=Path(NAMELIST_SELECTION[name]) if name in NAMELIST_SELECTION else (candidates[0] if len(candidates)==1 else None)
    try:resolved_store=Path(paths.resolve_cice_store())
    except FileNotFoundError:resolved_store=Path(paths.resolve_cice_store_target())
    row.update(output_root=str(root),history_present=(root/'history/daily').is_dir(),
        zarr_present=resolved_store.is_dir(),resolved_history_store=str(resolved_store),namelist_path=str(path) if path else '',
        namelist_sha256='',namelist_matches=False,provenance_reviewed=False,
        model_commit='',forcing_version='',restart_lineage='',spinup='1995–1999',
        regularisation_speed=REGULARISATION_SPEED.get(name,''),configuration_note='pending or ambiguous archive metadata')
    if path is not None:
        actual=wf.namelist_values(path); mismatches=[]
        for key in ['boundary_condition','lateral_drag','cs','cq','ktens','e_yieldcurve','e_plasticpot','kstrength','ndte']:
            value=actual.get(key)
            expected=row[key]
            try:
                if isinstance(expected,bool): value=value.lower()=='.true.' if value is not None and value.lower() in ['.true.','.false.'] else None
                elif isinstance(expected,(int,float)): value=float(value.lower().replace('d','e'))
                matched=value==expected
            except (AttributeError,ValueError): matched=False
            if not matched:mismatches.append(key)
        row.update(namelist_sha256=wf.file_sha256(path),namelist_matches=not mismatches,
                   configuration_note='matches expected scalar physics' if not mismatches else 'MISMATCH: '+', '.join(mismatches),
                   grid_file=actual.get('grid_file',''),form_factor_file=actual.get('f2_file',''),
                   atm_data_dir=actual.get('atm_data_dir',''),ocn_data_dir=actual.get('ocn_data_dir',''))
    rows.append(row)
manifest=pd.DataFrame(rows)
MANIFEST_PATH=EXPORT/'experiment_manifest.csv'
manifest.to_csv(MANIFEST_PATH,index=False)
provenance['manifest_sha256']=wf.file_sha256(MANIFEST_PATH)
SIGNATURE=wf.signature(provenance)
(EXPORT/'analysis_configuration.json').write_text(json.dumps(provenance,indent=2))
display(manifest)

## 2. Dates, units and definitions

I'll use exactly matched calendar-day support after checking native timestamps, leap days, missing days and the source sampling convention. Inner joins must not conceal dropped dates. The helpers reject duplicate/missing days and unreviewed units. FIA is concentration-weighted area when read from shuga; classified-cell extent is a different quantity. I'll reconcile the manuscript equation with the metric actually used.

Native AF2020 sampling is not automatically a daily observation. The optional interpolation below requires explicit review, bracketed source dates, no extrapolation and a maximum 20-day gap. It does not create independent daily observations. A comparison at native composite support requires its own interval treatment. I'll record which approach is used.

I'll check that the BS/AS bias signs and regional sums follow from the source products rather than repairing signs by eye. The previously quoted BS differences −46.3/−50.2 with observed mean 38.6 are impossible as model-minus-observation areas. The P95 difference 604.6−626.6 is −22.0 at displayed precision. Regional boundaries may have gaps/overlaps, so equality of summed sectors and the total is only expected after their partition is verified.

Centred classification needs surrounding history: at least five days for 9-of-11 and seven days for the 15-day rolling window. I'll also quantify valid exact-zero speed cells before accepting the classifier's zero-speed exclusion. The second notebook contains that audit.

## 3. Processing recipes — run through PBS, not automatically

In [ ]:
if float(CFG['aice_thresh']) != 0.15:
    raise ValueError('The current shuga metrics CLI has no --aice-thresh option; use explicit ClassificationSpec for a non-default threshold')
commands=[]
for name in ACTIVE_EXPERIMENTS:
    common=['--sim-name',name,'--experiment-group',GROUP,'--start-date',START,'--end-date',END,
        '--project',CFG['project'],'--user',CFG['user'],'--hemisphere','SH',
        '--afim-output-root',str(BANK)]
    classification=['python3','scripts/classification/classify.py',*common,'--ice-type','FI',
        '--grid-type',CFG['grid_type'],'--ispd-thresh',str(CFG['ispd_thresh']),'--aice-thresh',str(CFG['aice_thresh']),
        '--methods','raw,binary-days,rolling-mean','--bin-window',str(CFG['bin_window']),'--bin-min-days',str(CFG['bin_min_days']),
        '--roll-window',str(CFG['roll_window']),'--skip-history-conversion']
    metrics=['python3','scripts/metrics/metrics.py',*common,'--ice-type','FI','--grid-type',CFG['grid_type'],
        '--ispd-thresh',str(CFG['ispd_thresh']),'--methods',','.join(ROBUSTNESS_METHODS) if RUN_ROBUSTNESS else METHOD,'--bin-window',str(CFG['bin_window']),'--bin-min-days',str(CFG['bin_min_days']),
        '--roll-window',str(CFG['roll_window']),'--metric-groups','fi_core,fi_spatial,fi_regional,fi_summary','--update-missing-only']
    commands.extend([shlex.join(classification),shlex.join(metrics)])
commands.extend([shlex.join(['python3','scripts/comparisons/FIA_statistics_tables.py',
    '--experiment-group',GROUP,'--sim-names',' '.join(ACTIVE_EXPERIMENTS),
    '--start-date',OBS_START,'--end-date',OBS_END,'--classification',METHOD,'--grid-type',CFG['grid_type'],
    '--afim-output-root',str(BANK),'--project',CFG['project'],'--user',CFG['user'],
    '--af2020-fia-store',str(OBS_FIA_PATH),'--table-root',str(EXPORT/'shuga_FIA_tables'),
    '--ispd-thresh',str(CFG['ispd_thresh']),'--bin-window',str(CFG['bin_window']),
    '--bin-min-days',str(CFG['bin_min_days']),'--roll-window',str(CFG['roll_window']),
    '--min-edge-trim-days',str(MIN_EDGE_TRIM_DAYS),
    '--growth-months',*[str(m) for m in season_months(GROWTH_WINDOW)],
    '--retreat-months',*[str(m) for m in season_months(RETREAT_WINDOW)]]),
    '# FIP spatial support/remapping is reviewed and computed in LD-performance.ipynb.'])
(EXPORT/'processing_commands.sh').write_text('\n'.join(commands)+'\n')
print(EXPORT/'processing_commands.sh')
print('Run these from the standalone shuga repository, after loading the analysis environment.')

Archive staging and NetCDF-to-Zarr conversion are separate operations. From shuga, inspect `python3 scripts/admin/archive.py --verbose`, then stage completed segments with `--send-to-archive`. The explicit case map is free-slip → free-slip-no-drag, free-slip01 → Cs-high-ktens-zero, free-slip02 → no-slip-LFI-match. Verify actual source locations first. Convert history before using `--skip-history-conversion`.

`--update-missing-only` does not validate old values after definitions, periods or source files change. I'll rebuild the affected products and record their versions rather than mixing legacy SeaIceToolbox values with new stores. I'll use actual cell areas, never the nominal 25 km fallback. Shared grid assets remain outside experiment groups.

## 4. Analytical form functions

I'll plot only the retained static and quadratic functions, using verified archived coefficients and the executed static regularisation speed. I did not find an equivalent shuga analytical plotter; the small PyGMT plot below remains local. I'll check the equations and common phi units against the executable before enabling it. These curves do not establish the realised drag stress or an activation regime.

In [ ]:
if RUN_ANALYTICAL:
    if not FORM_FUNCTION_DEFINITION_VERIFIED or not FORM_FUNCTION_UNITS:
        raise ValueError("Review executed equations and phi units before plotting")
    import pygmt
    speed=np.linspace(0,1.1e-3,900)
    curves={}; rows=[]
    for name in FAMILIES['static']+FAMILIES['quadratic']:
        row=manifest.set_index('experiment').loc[name]
        if not bool(row.namelist_matches):raise ValueError(f"{name}: verify archived namelist first")
        if name in FAMILIES['static']:
            u0=float(row.regularisation_speed)
            if not np.isfinite(u0) or u0<=0:raise ValueError(f"{name}: verify positive u0")
            phi=float(row.cs)/(speed+u0)
        else:
            phi=float(row.cq)*speed
        if not np.isfinite(phi).all():raise ValueError("Non-finite form-function curve")
        curves[name]=phi
        rows.extend(dict(experiment=name,speed_m_s=u,phi=p,phi_units=FORM_FUNCTION_UNITS) for u,p in zip(speed,phi))
    export_table(pd.DataFrame(rows),'analytical_form_functions',list(curves),(START,END))
    ymax=max(float(v.max()) for v in curves.values())*1.05
    fig=pygmt.Figure()
    fig.basemap(region=[0,float(speed.max()),0,ymax],projection='X16c/9c',
        frame=['WSen+tStatic and quadratic form functions','xaf+lIce speed (m/s)',f'yaf+lPhi ({FORM_FUNCTION_UNITS})'])
    colors=['blue','orange','darkgreen','red','purple','black']
    for (name,phi),color in zip(curves.items(),colors):fig.plot(x=speed,y=phi,pen=f'1.5p,{color}',label=name)
    fig.plot(x=[CFG['ispd_thresh']]*2,y=[0,ymax],pen='1p,gray,--',label='FI speed threshold')
    fig.legend(position='JTR+jTR+o0.2c',box='+gwhite+p0.5p')
    path=EXPORT/'analytical_form_functions_Cs_Cq.png';fig.savefig(path)
    register(path,'PyGMT analytical form functions',list(curves),(START,END))

## 5. FIA: definitions, common samples and climatology

I'll reuse the schema/statistics functions in shuga's `FIA_statistics_tables.py`, `skill_stats`, and its monthly statistical reducer. Figures use a small FIA-labelled PyGMT renderer; shuga's SIA/SIT plotting functions hard-code different axis labels and must not be presented as FIA unchanged.

I'll require explicit area units before using shuga's schema coercion, reject negative areas and duplicate region labels, and require complete model daily coverage. Region labels and sampled dates must align exactly; there is no nearest-date substitution.

`FIA_OBS_MODE='linear-daily'` preserves the earlier notebook policy: explicitly reviewed linear interpolation, bracketed observations and a maximum 20-day gap. It does not create independent daily observations. The optional `native-dates` mode selects model calendar days at the native observation timestamps without interpolation. Neither choice by itself reproduces a composite's temporal interval, so I'll review that support separately before calculating skill.

Native AF2020 summaries are exported separately. Monthly means and their interannual standard deviations are computed using the same reducer shuga uses for thickness; the numeric reduction is generic and FIA units are retained. Partial endpoint months/years remain labelled. The underlying FIA CLI averages each source at its own timestamps, so its recipe tables are a separate descriptive product, not interchangeable with these matched-sample skill statistics.

In [ ]:
# Import shuga's existing comparison functions without running its CLI or retaining legacy sys.path edits.
FIA_DATA={};FIA_RESULTS=None;FIA_SUMMARIES=None;FIA_ANNUAL=None;obs=None;fia_api=None

def load_fia_api():
    import runpy
    script=D_shuga/'scripts/comparisons/FIA_statistics_tables.py'
    if not script.is_file():raise FileNotFoundError(script)
    previous=sys.path.copy()
    try:api=runpy.run_path(str(script),run_name='shuga_fia_notebook')
    finally:sys.path[:]=previous
    provenance['shuga_fia_script_sha256']=wf.file_sha256(script)
    provenance['shuga_reduction_files']={relative:wf.file_sha256(Path(shuga.__file__).parent/relative) for relative in ['metrics/skill.py','metrics/temporal.py','plotting/cice.py']}
    return api

def model_fia(name,method=METHOD):
    ds=load(name,'metrics',['FIA','FIA_by_region'],method)
    converted=xr.Dataset({v:wf.area_to_kkm2(audited(ds[v],name)) for v in ['FIA','FIA_by_region']})
    out=fia_api['_coerce_fia_time_region'](converted,source_name=name,require_regional=True)
    labels=pd.Index(out.region.values)
    if labels.has_duplicates:raise ValueError(f"{name}: duplicate region labels")
    wf.daily_check(out,START,END)
    return out.assign_coords(time=pd.DatetimeIndex(out.time.values).normalize())

def match_fia_observation(data):
    # Exact calendar days only; no nearest-date substitution.
    data=data.sel(time=obs.time)
    data,reference=xr.align(data,obs,join='exact')
    wf.finite_check(data);wf.finite_check(reference)
    if bool(((data<0)|(reference<0)).any().item()):raise ValueError('Negative FIA')
    return data,reference

def paired_fia_statistics(data,reference,name,method):
    from shuga.metrics.skill import skill_stats
    data,reference=xr.align(data,reference,join='exact')
    rows=[]
    for region in data.region.values:
        m=data.sel(region=region);o=reference.sel(region=region)
        wf.finite_check(m);wf.finite_check(o)
        with np.errstate(divide='ignore',invalid='ignore'):stats=skill_stats(m.values,o.values)
        rows.append(dict(experiment=name,method=method,region=str(region),n_samples=m.sizes['time'],
            mean=float(m.mean()),min=float(m.min()),max=float(m.max()),p95=float(m.quantile(.95)),
            obs_mean=float(o.mean()),bias=stats['Bias'],mae=stats['MAE'],rmse=stats['RMSE'],correlation=stats['Corr'],
            units='10^3 km^2',observation_support=FIA_OBS_MODE))
    return pd.DataFrame(rows)

if RUN_FIA:
    if not FIA_DEFINITION_VERIFIED or not OBS_FIA_TEMPORAL_SUPPORT_VERIFIED:
        raise ValueError('Review FIA definition and observation/model temporal support')
    fia_api=load_fia_api()
    with open_product(OBS_FIA_PATH) as source:
        obs_native=wf.area_to_kkm2(audited(source.FIA,'AF2020')).load()
    if 'region' not in obs_native.dims:raise ValueError('Observed FIA requires reviewed region labels')
    dates=pd.DatetimeIndex(obs_native.time.values)
    if dates.has_duplicates or dates.normalize().has_duplicates or not dates.is_monotonic_increasing:
        raise ValueError('Observed dates must be sorted and unique by calendar day')
    obs_native=obs_native.assign_coords(time=dates.normalize())
    if pd.Index(obs_native.region.values).has_duplicates:raise ValueError('Duplicate observed region labels')
    obs_native=fia_api['_region_order_present'](obs_native)
    if FIA_OBS_MODE=='linear-daily':
        if not OBS_FIA_INTERPOLATION_VERIFIED:raise ValueError('Review explicit daily interpolation before proceeding')
        obs=wf.daily_observation(obs_native,OBS_START,OBS_END,max_gap_days=20)
    elif FIA_OBS_MODE=='native-dates':
        obs=obs_native.sel(time=slice(OBS_START,OBS_END))
        if not obs.sizes.get('time',0):raise ValueError('No native observation dates in the comparison period')
    else:raise ValueError('Choose linear-daily or native-dates')
    wf.finite_check(obs)
    if bool((obs<0).any().item()):raise ValueError('Negative observed FIA')
    summaries=[fia_api['summarise_fia'](obs,source=FIA_OBS_MODE,sim_name='AF2020',
        start_date=OBS_START,end_date=OBS_END,min_edge_trim_days=0)]
    tables=[]
    for name in ACTIVE_EXPERIMENTS:
        data,reference=match_fia_observation(model_fia(name).sel(time=slice(OBS_START,OBS_END)))
        data=data.compute();FIA_DATA[name]=data
        tables.append(paired_fia_statistics(data,reference,name,METHOD))
        summaries.append(fia_api['summarise_fia'](data,source='simulation',sim_name=name,
            start_date=OBS_START,end_date=OBS_END,min_edge_trim_days=MIN_EDGE_TRIM_DAYS))
        COVERAGE.append(dict(experiment=name,diagnostic='FIA',n_samples=data.sizes['time'],support=FIA_OBS_MODE,status='passed'))
    FIA_RESULTS=pd.concat(tables,ignore_index=True)
    FIA_SUMMARIES=pd.concat(summaries,ignore_index=True)
    FIA_SUMMARIES['observation_support']=FIA_OBS_MODE
    SIGNATURE=wf.signature(provenance)
    export_table(FIA_RESULTS,'FIA_skill',ACTIVE_EXPERIMENTS)
    export_table(FIA_SUMMARIES,'FIA_shuga_statistics',ACTIVE_EXPERIMENTS)
    native_summary=fia_api['summarise_fia'](obs_native.sel(time=slice(OBS_START,OBS_END)),
        source='native observation',sim_name='AF2020',start_date=OBS_START,end_date=OBS_END,min_edge_trim_days=0)
    export_table(native_summary,'FIA_native_AF2020_statistics',['AF2020'])
    display(FIA_RESULTS)
else:print('Pending: enable RUN_FIA after reviewing the data definitions and sampling policy')

In [ ]:
# Replot reduced FIA series without reopening model histories.
if RUN_FIA_PLOTS:
    if not FIA_DATA:raise ValueError('Run FIA reductions first')
    import pygmt
    from shuga.plotting.cice import CICEPlotter
    cases=[name for name in FIA_PLOT_CASES if name in FIA_DATA]
    if not cases:raise ValueError('No selected FIA plot cases were analysed')
    series={name:FIA_DATA[name].sel(region='circum_antarctic').to_series() for name in cases}
    series['AF2020']=obs.sel(region='circum_antarctic').to_series()
    # This numeric monthly reducer is also used by shuga's SIT plotter; no units are converted here.
    climatologies={name:CICEPlotter.sit_monthly_climatology(s,envelope='std') for name,s in series.items()}
    frames=[]
    for name,table in climatologies.items():
        frame=table.copy();frame['experiment']=name;frame['units']='10^3 km^2';frames.append(frame)
    export_table(pd.concat(frames,ignore_index=True),'FIA_monthly_climatology',list(series))
    upper=max(float(t[['mean','upper']].max().max()) for t in climatologies.values())*1.1
    finite_lowers=[float(t.lower.min()) for t in climatologies.values() if t.lower.notna().any()]
    lower=min([0.]+finite_lowers)
    fig=pygmt.Figure()
    fig.basemap(region=[.5,12.5,lower,upper],projection='X17c/9c',
        frame=['WSen+tFIA monthly climatology','xa1+lMonth','yaf+lFast-ice area (10@+3@+ km@+2@+)'])
    colors=['blue','orange','darkgreen','purple','red','brown','cyan4']
    for index,(name,table) in enumerate(climatologies.items()):
        color='black' if name=='AF2020' else colors[index%len(colors)]
        # Draw interannual-spread envelopes only over contiguous finite month segments.
        good=np.isfinite(table.lower)&np.isfinite(table.upper)
        for _,part in table[good].groupby((~good).cumsum()):
            if len(part)>1:fig.plot(x=np.r_[part.month,part.month[::-1]],y=np.r_[part.upper,part.lower[::-1]],close=True,fill=f'{color}@88',pen='0p')
        fig.plot(x=table.month,y=table['mean'],pen=f"1.5p,{color}"+(',--' if name=='AF2020' else ''),label=name)
    fig.legend(position='JTR+jTR+o0.2c',box='+gwhite+p0.5p')
    path=EXPORT/f'FIA_climatology_{OBS_START}_{OBS_END}.png';fig.savefig(path)
    register(path,'PyGMT FIA monthly climatology',list(series),notes=f'{FIA_OBS_MODE}; shading: interannual monthly standard deviation; partial endpoint months retained')

In [ ]:
# Reuse shuga annual extrema and full-window seasonal-rate diagnostics.
if FIA_DATA:
    from shuga.metrics.temporal import compute_extrema_table
    annual=[]
    for name,data in {**FIA_DATA,'AF2020':obs}.items():
        for region in data.region.values:
            table=compute_extrema_table(data.sel(region=region),variable='FIA',sim_name=name,
                include_mean=False,include_overall=False,growth_window=GROWTH_WINDOW,
                retreat_window=RETREAT_WINDOW,require_full_rate_window=True,
                rate_min_points=3 if FIA_OBS_MODE=='native-dates' else 20)
            table['region']=str(region);table['observation_support']=FIA_OBS_MODE
            table['complete_calendar_year']=[
                pd.Timestamp(row.start_date)==pd.Timestamp(int(row.year_start),1,1)
                and pd.Timestamp(row.end_date)==pd.Timestamp(int(row.year_start),12,31)
                for row in table.itertuples()]
            annual.append(table)
    FIA_ANNUAL=pd.concat(annual,ignore_index=True)
    export_table(FIA_ANNUAL,'FIA_annual_extrema_and_rates',list(FIA_DATA)+['AF2020'])

## 6. Matched effects, geometry and rheology

All differences are **target minus reference** as labelled. Boundary-only: free-slip-no-drag minus no-slip-LFI-match. Drag effect: active Cs/Cq minus free-slip-no-drag. Added tensile term: Cs-high minus Cs-high-ktens-zero, supplemented by the full 0/0.05/0.1/0.2 sequence. I'll interpret these as model contrasts and evaluate observational skill separately.

The geometry cases are matched in namelist physics apart from the form-factor source. Their responses need not add linearly under maximum-combined geometry. Zero added tensile strength retains compressive/shear rheology; the static tensile sequence does not test quadratic tensile dependence or provide a full factorial design.

In [ ]:
if FIA_RESULTS is not None:
    for label,(target,reference) in PAIRS.items():
        if target not in FIA_DATA or reference not in FIA_DATA:
            print(f'Pending {label}: {target} / {reference}');continue
        table=paired_fia_statistics(FIA_DATA[target],FIA_DATA[reference],target,METHOD).rename(columns={'obs_mean':'reference_mean','bias':'mean_difference'})
        table.insert(0,'target',target);table.insert(1,'reference',reference)
        export_table(table,'contrast_'+label,[target,reference]);display(table)
    for label,cases in {'coefficient':FAMILIES['static']+FAMILIES['quadratic'],
        'geometry':['cst-drag','GIB-drag','Cs-high'],'tensile':TENSILE_CASES,
        'other_rheology':FAMILIES['rheology']}.items():
        present=[case for case in cases if case in FIA_DATA]
        if present:export_table(FIA_RESULTS[FIA_RESULTS.experiment.isin(present)],label+'_FIA',present)
    # Sector/total mismatch is a diagnostic, not an instruction to fix signs.
    for name,data in FIA_DATA.items():
        sectors=data.drop_sel(region='circum_antarctic');total=data.sel(region='circum_antarctic')
        table=wf.audit_region_totals(sectors,total,REGIONAL_SUM_TOLERANCE)
        export_table(table,'region_total_audit_'+name,[name])
        if REGIONAL_PARTITION_VERIFIED and not table.within_tolerance.all():
            raise ValueError(f'{name}: a verified partition does not reconcile with total area')

## 7. Seasonal growth and classification robustness

I'll use `shuga.metrics.temporal.compute_extrema_table` for annual extrema and seasonal rates. The declared windows are April–July growth and December–March retreat, with full-window coverage required. These differ from the FIA CLI's default seasons; the generated recipe passes the chosen months explicitly. Partial endpoint years remain in the annual inventory and do not acquire a rate for an unsupported season.

I'll repeat the matched FIA comparison for the available raw, binary-days and rolling-mean metrics. This tests sensitivity without another model run. Parameter selection and scoring on these same years remain in-sample evidence. Interannual envelopes and adjacent pixels are not independent validation samples.

In [ ]:
ROBUSTNESS_RESULTS=None
if FIA_ANNUAL is not None:
    rate_columns=[c for c in FIA_ANNUAL if c.startswith(('growth_','retreat_'))]
    export_table(FIA_ANNUAL[['sim_name','region','period','observation_support']+rate_columns],
        'FIA_seasonal_growth_retreat',list(FIA_DATA)+['AF2020'])
if RUN_ROBUSTNESS:
    if not RUN_FIA:raise ValueError('Run FIA first')
    tables=[]
    for method in ROBUSTNESS_METHODS:
        for name in ACTIVE_EXPERIMENTS:
            data,reference=match_fia_observation(model_fia(name,method).sel(time=slice(OBS_START,OBS_END)))
            tables.append(paired_fia_statistics(data.compute(),reference,name,method))
    ROBUSTNESS_RESULTS=pd.concat(tables,ignore_index=True)
    export_table(ROBUSTNESS_RESULTS,'classification_robustness',ACTIVE_EXPERIMENTS)

## 8. Manuscript exports and scientific review

I'll review the companion notebook's standard shuga FIP differences and PyGMT maps; mechanics and actual E/N-face stresses; annual occurrence and FI-conditioned tendencies; reference-anchor distances; pack/SH-total metrics; and contemporaneous point-thickness comparisons where observations overlap the model period. FIP retains its own native sampling, and point scores have explicitly conditional support; I'll not equate those with FIA's selected sampling policy.

The inventory below records whether companion export records exist. Existence is not scientific approval, and a shared file path does not prove that both kernels used identical input versions. I'll check dates, signatures, settings, source provenance and scientific limitations before marking `companion_diagnostics_reviewed` complete. Companion export filenames are not rewritten by this notebook.

### Writing and figure checks I'll finish here

- I'll reconcile the September scope changes with the TeX, retained equations/macros, experiment table and figure list. Linear and strain-blend families, waves/tides and the other out-of-scope extensions should not remain in the scientific results by accident.
- I'll check the Australian-sector form-factor file paths against the Bellingshausen labels/statistics, and correct the actual region, geometry annotations and captions together. I'll verify that VOL and WS use their intended PNG resources.
- I'll replace the analytical form-function, FIA and mechanical comparison placeholders with checked exports. I'll preserve the intended two-panel sector comparisons, titles, coordinates and colour bars; no blanket image cropping. I'll consolidate overlapping FIP sections and move repetitive sector maps to the supplement if useful.
- I'll write the parameter trade-offs, seasonal/regional errors, geometry and forcing limitations, and implications for coupled models into the discussion. Parameter selection and evaluation on these same years remain in-sample evidence.
- I'll finish conclusions and quantitative abstract wording only after the tables reconcile. I'll check authorship, dataset attribution, funding/disclaimer, accessible code/data versions, and the final questions for co-authors. I'll assemble `copernicus.cls`, `refs_bbt.bib`, all figure resources and style dependencies, then compile and inspect the manuscript and supplement before Alex's review.

The three new contrasts are not a complete boundary × drag × tensile factorial design. The tensile sequence tests the static closure; it does not establish quadratic-closure tensile dependence. Zero added tensile strength retains other compressive/shear resistance, and low speed alone does not demonstrate mechanical coherence.

### Working abstract — wording to complete after the comparisons

Antarctic landfast sea ice is an important component of the coastal cryosphere, influencing air–sea exchange, polynya activity, coastal access and habitat. Its mechanical anchors are often unresolved in global sea-ice models. Here I introduce and evaluate a lateral-drag parameterisation in CICE6, separating geometric form factors from speed-dependent form functions and representing coastal and grounded-iceberg resistance on C-grid velocity points. Building on the coastal-drag formulation of Liu et al. (I'll check the publication year against the bibliography), I use a global quarter-degree stand-alone configuration to compare matched boundary conditions and drag activation, coefficient choices, geometric contributions and rheological sensitivity. **I'll add verified quantitative results and limitations here.** The assessment will establish how far the formulation represents Antarctic fast-ice anchoring without adding grounded icebergs to the land mask, and what remains to be tested before coupled-model use.


In [ ]:
# Presence of a companion artifact is inventory evidence, not scientific approval.
companion_inventory=[]
for filename in ['supporting_artifacts.json','supporting_review.json']:
    path=EXPORT/filename
    companion_inventory.append(dict(path=str(path),present=path.is_file(),
        sha256=wf.file_sha256(path) if path.is_file() else ''))
export_table(pd.DataFrame(companion_inventory),'companion_export_inventory',[],(START,END))
REVIEW={
    'source_and_restart_provenance_reviewed':PROVENANCE_REVIEWED,
    'all_retained_experiments_selected':set(ACTIVE_EXPERIMENTS)==set(EXPERIMENTS),
    'namelist_physics_matched':bool(manifest.namelist_matches.all()),
    'geometry_comparison_provenance_checked':GEOMETRY_MATCH_VERIFIED,
    'FIA_definition_and_recomputed_numbers':FIA_DEFINITION_VERIFIED and FIA_RESULTS is not None,
    'observation_temporal_support_reviewed':OBS_FIA_TEMPORAL_SUPPORT_VERIFIED,
    'all_three_new_runs_analysed':all(n in FIA_DATA for n in ['free-slip-no-drag','no-slip-LFI-match','Cs-high-ktens-zero']),
    'classification_robustness_checked':ROBUSTNESS_RESULTS is not None,
    **MANUSCRIPT_CHECKS,
}
(EXPORT/'analysis_configuration.json').write_text(json.dumps(provenance,indent=2))
(EXPORT/'FIA_coverage.json').write_text(json.dumps(COVERAGE,indent=2))
(EXPORT/'manuscript_artifacts.json').write_text(json.dumps(ARTIFACTS,indent=2))
(EXPORT/'manuscript_review.json').write_text(json.dumps({k:bool(v) for k,v in REVIEW.items()},indent=2))
display(pd.DataFrame({'item':REVIEW.keys(),'complete':REVIEW.values()}))
for ds in OPEN.values():ds.close()
OPEN.clear()
if FINAL_EXPORT and not all(REVIEW.values()):raise RuntimeError('Scientific/manuscript review is incomplete')